# Calculate Harmonic Parameters from Sentinel-1 Backscatter Virtual Zarr

For operational flood mapping operations, the area of interest may be significantly smaller than a single Equi7 Grid tile. Fetching data from (non-Cloud Optimized) GeoTIFFs stored in a STAC catalogue requires downloading an entire tile-size image for each timestamp of interest, when only a small portion of each image is actually needed. This overhead is not so burdensome when analysing a single flood event, as only a handful of images are needed, but recalculation of harmonic parameters requires data from at least a full year of observations, which can be dozens of images.

We can avoid this overhead by using a virtual Zarr dataset created with [kerchunk](https://fsspec.github.io/kerchunk/). Because the virtual Zarr provides a reference from the coordinates of the S1 datacube to the byte ranges of the corresponding GeoTIFF chunks, we can download and read only the portions of each image that intersect our area of interest, and avoid downloading entire images. This is particularly efficient when combined with Dask, which can load and process those chunks in parallel. To read more about virtual Zarr datasets, see [kerchunk](https://fsspec.github.io/kerchunk/) and [virtualizarr](https://virtualizarr.readthedocs.io/en/latest/)

Let's begin by downloading the virtual Zarr. This could also be hosted on object storage, eliminating the need to download the entire reference file.

In [ ]:
zip_url_root = "https://git.geo.tuwien.ac.at/public_projects/rs/s1-virtualzarr/-/raw/main/"
zip_filename = "SIG0_S1_2022-2023_EU020M.parq.zip"
zip_path = "/tmp/" + zip_filename

import os

if not os.path.exists(zip_path):
    import urllib.request

    urllib.request.urlretrieve(zip_url_root + zip_filename, zip_path)

In [ ]:
import dask
import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar
from dask_flood_mapper.harmonic_params import create_harmonic_parameters_zarr
from dask_flood_mapper.vzarr import open_s1_datacube
from dask_flood_mapper.vzarr.utils import get_bbox_from_tile_cube
from matplotlib import pyplot as plt

pbar = ProgressBar()
pbar.register()

As an example we will select only a small region of interest contained in the Zingst case study. 

We will now open up the Sentinel 1 Virtual Zarr and select the parts we need.

In [ ]:
y_chunks = 600
s1_ds = open_s1_datacube(
    zip_path,
    chunks={
        "X": 15000,
        "Y": y_chunks,
        "polarization": 1,
        "obs": 10,
        "orbit": 1,
        "tile": 1,
    },
)

In [ ]:
from pyproj import Transformer

minlon, maxlon = 12.3, 13.1
minlat, maxlat = 54.3, 54.6
((minx, maxx), (miny, maxy)) = Transformer.from_crs(
    "EPSG:4326", "EPSG:27704", always_xy=True
).transform([minlon, maxlon], [minlat, maxlat])
bounding_box = [minx, miny, maxx, maxy]
bounding_box

In [ ]:
region_ds = get_bbox_from_tile_cube(
    s1_ds, bounding_box, y_chunk_size=y_chunks
).sel(polarization="VV", drop=True)
region_ds

We only need parameters for orbits that cross our actual time of interest, so we can filter those out as well.

In [ ]:
region_ds.time.load()
flood_times = (region_ds.time >= np.datetime64("2023-10-11T00:00:00")) & (
    region_ds.time < np.datetime64("2023-10-26T00:00:00")
)
harmpar_times = (region_ds.time >= np.datetime64("2022-10-11T00:00:00")) & (
    region_ds.time < np.datetime64("2023-10-11T00:00:00")
)

In [ ]:
flood_ds = region_ds.where(flood_times, drop=True)
# we can shortcut some calculations later by setting data to NaN for any orbits where we don't have flood data for a tile
harmpar_ds = region_ds.where(harmpar_times, drop=True).where(
    flood_times.any(["obs"])
)

In [ ]:
flood_ds

In [ ]:
harmpar_ds

## Calculate Harmonic Parameters

This function fits sine and cosine functions known as harmonic oscillators to each pixel of the Sentinel 1 $\sigma^0$ datacube. These seasonally varying curves can then be extracted from time series. What is left is the noise or transient events, for example flood events, superimposed on the seasonal trend.

Because the virtual Zarr dataset is already structured along tile and orbit dimensions, and chunked along the Y dimension, Dask can efficiently load only the data needed for each tile, orbit, and chunk, and process them in parallel.

In [ ]:
hpar_dc = create_harmonic_parameters_zarr(
    harmpar_ds.sel(orbit=flood_ds.orbit)
    .isel(orbit=[1])
    .chunk({"Y": y_chunks, "orbit": 1, "obs": -1, "tile": 1}),
    min_nobs=10,
)
hpar_dc

The result of the last cell is lazy. Finally, we can mosaic the constituent tiles together to get a complete harmonic parameter dataset for our region of interest.

In [ ]:
hpar_mosaic = xr.combine_by_coords(
    [
        hpar_dc.sel(tile=i, drop=True).set_xindex("Y").set_xindex("X")
        for i in hpar_dc.tile.values
    ]
).sel(X=slice(minx, maxx), Y=slice(maxy, miny))
hpar_mosaic

We probably want to use these harmonic parameters more than once without recalculating them, so we will save them to a Zarr store after computing them.

In [ ]:
with dask.config.set(scheduler="threading"):
    hpar_mosaic.load().chunk("auto").rio.set_spatial_dims(
        x_dim="X", y_dim="Y"
    ).rio.write_crs("EPSG:27704").to_zarr("hpar_mosaic.zarr")

Let's map the harmonic parameters as a sanity check:

In [ ]:
hpar_mosaic["M0"].plot.imshow(
    cmap="viridis",
    vmin=-20,
    vmax=0,
    col="orbit",
    col_wrap=3,
)
plt.savefig("harmonic_parameters_M0.png", dpi=300)

In [ ]:
plt.close()
hpar_mosaic.isel(orbit=0)[["S1", "S2", "S3"]].to_dataarray(
    dim="param"
).plot.imshow(
    cmap="viridis",
    vmin=-2,
    vmax=2,
    col="param",
)
plt.savefig("harmonic_parameters_sine.png", dpi=300)

In [ ]:
from matplotlib import pyplot as plt

hpar_mosaic.isel(orbit=0)[["C1", "C2", "C3"]].to_dataarray(
    dim="param"
).plot.imshow(cmap="viridis", vmin=-2, vmax=2, col="param")
plt.savefig("harmonic_parameters_cosine.png", dpi=300)

We can now use the harmonic parameters to generate predicted $\sigma^0$ values for the flood period.

In [ ]:
t = flood_ds.time.dt.dayofyear
n = 365
y = (
    hpar_dc.M0
    + hpar_dc.C1 * np.cos(2 * np.pi * t / n)
    + hpar_dc.S1 * np.sin(2 * np.pi * t / n)
    + hpar_dc.C2 * np.cos(2 * np.pi * t / n)
    + hpar_dc.S2 * np.sin(2 * np.pi * t / n)
    + hpar_dc.C3 * np.cos(2 * np.pi * t / n)
    + hpar_dc.S3 * np.sin(2 * np.pi * t / n)
)

## Fit Harmonic Function to Original Data

Finally, we merge the two datasets and superimpose the fitted harmonic function on the raw sigma nought data. 

In [ ]:
xr.merge([y.rename("pred"), flood_ds.sig0]).squeeze().hvplot(x="time")

## Integrate into Flood Mapping Workflow

In order to integrate the calculated harmonic parameters into the flood mapping workflow (see [notebook 3](03_flood_map.ipynb)), we must simply read from the Zarr store and reproject from the Equi7 grid to WGS84. Replace the code in notebook 3's section "Harmonic Parameters" with the following:

In [ ]:
hpar_dc = xr.open_zarr(
    "hpar_mosaic.zarr",
    chunks={"Y": y_chunks, "X": 15000, "orbit": 1, "tile": 1},
).chunk({"Y": y_chunks, "X": 15000, "orbit": 1, "tile": 1})
hpar_dc = (
    hpar_dc.rio.set_spatial_dims(x_dim="X", y_dim="Y")
    .rio.reproject("EPSG:4326")
    .rename({"Y": "latitude", "X": "longitude"})
)
hpar_dc